## Splash Test 💧💧💧

<div class="alert alert-block alert-success">

## Part 3: Executable Code & Running in Terminal

As mentioned in Part 2, we will go over the code in `shallow_splash.py` in detail here, but then give instructions for how to run in terminal. 

This is because the Cubed-Sphere format requires at least 6 CPU cores to run, which is not possible to achieve in a single-core Jupyter notebook. 

</div>

<div class="alert alert-warning">

## $\texttt{PADDLE}$ Splash Executable Code 

Since we moving to more complex 3D simulations, the code set up will be a bit different, but we will describe it below. 

<div class="alert alert-info">

## First, import the necessary packages. 

Here, there are few key differences to things we imported in previous tutorials. 

The first is that for shallow splash **density = geopotential**. This is because density is assumed constant in the shallow splash, so we assign geopotential to the density slot in the tensor. 

This is also the first time we import the keys for the velocity components (kIV2 and kIV3), which we use to explicitly set the initial condition where velocities are 0. The other primitive variables (kIV1) is always 0 for a single layer in the vertical direction.

We also import snapy functions that are essential to the Cubed-Sphere geometry, which we will describe more in detail below. 

```python

# Allows the input .yaml and output directory to be defined in commmand line
# Described below
import argparse

# Utilized to create output directories
import os

# Import numpy in order to get pi 
import numpy as np

# Torch is the basic computational backend, equivalent in some ways to numpy but can work on CPU and GPU
import torch

# Read and parse YAML configuration files
import yaml

# Here we import the main module (Mesh) and options (MeshOptions) used in all PADDLE simulations
# Note that these are similar to MeshBlock and MeshBlockOptions used in previous tutorials
# and the named locations of geopotential (where, for shallow splash the geopotential lives in the density slot of the tensor)
# and the horizontal velocities (kIV2 and kIV3)
from snapy import Mesh, MeshOptions
from snapy import kIDN as kIGP
from snapy import kIV2, kIV3

# We also import modules needed for the cubed-sphere geometry, described in detail below. 
from snapy.coord import cs_ab_to_lonlat, get_cs_face_name

<div class="alert alert-info">

## Custom user output variables 

Here, we opt to not define at custom outputs (like temperature) since we are only interested in how the geopotential evolves over the sphere. (which is already included in the primitive variable outputs)

<div class="alert alert-info">

# Let torch know if you are using CPU or GPU

`MeshOptions` reads `BACKEND`, `DEVICE`, and `DEVICE_ID` from the environment. The `Mesh` object initializes and owns communication internally.

```python

options = MeshOptions.from_yaml(input_file, verbose=False)
device = torch.device(options.device_str())

<div class="alert alert-info">

## Function to initialize block settings

In previous notebooks, we set up the block manually, where the block stores all physical values inside a pre-defined grid. Here, we make a function that will be useful as we move to more complex models. 

This function loads in relevant configuration properties, loads in the 3D grid, and then sets the initial hydrodynamic condition at each point in the 3D grid. 

```python

# Function takes in the bloc, the configuration dictionary (made from the yaml file), and the devices (cpu/gpu locations) and returns a dictionary
def initialize_block(
    block, config: dict, device: torch.device
) -> dict[str, torch.Tensor]:

    # Here, we load in the problem variables, defined in the yaml file in Part 2
    phi = float(config["problem"]["phi"])
    dphi = float(config["problem"]["dphi"])
    radius = float(config["problem"]["radius"])

    # Fetch the coordinate properties of the block
    # We will need this to generate the meshgrid we will define our initial condition of the simulation on top of 
    coord = block.module("coord")
    layout = block.get_layout()
    # Get specific face of the cubed-sphere mapped to a specific parallel worker process
    _, _, face_id = layout.loc_of(layout.options.rank())
    face = get_cs_face_name(face_id)

    # returns a 1D tensor with length = number of cells in x2 direction + 2 x ghost zones per face of the cube
    # As defined in the .yaml file, this would be 48 cells + 2 x 3 ghost zones = 54 
    beta, alpha, r_planet = torch.meshgrid(
        coord.buffer("x3v"), coord.buffer("x2v"), coord.buffer("x1v"), indexing="ij"
    )
    # Helper function that turns the two anglular and one radius dimension into latitude, longitude to help set initial condition below
    _, lat = cs_ab_to_lonlat(face, alpha, beta)

    # The number of points in each dimension
    # This is identical to the number of cells 
    nc3 = coord.buffer("x3v").shape[0]
    nc2 = coord.buffer("x2v").shape[0]
    nc1 = coord.buffer("x1v").shape[0]

    # Hardcode the number of variables (4 = number of primitive variables in shallow splash -> geopotential, 3 velocities)
    nvar = 4

    # Define a 4D tensor storing primitive variables at each simulation coordinate
    # In athena++, w signifies the primitive variables 
    w = torch.zeros((nvar, nc3, nc2, nc1), device=device)
    gc_dist = r_planet * (np.pi / 2.0 - lat)

    # Create geo-potential initial condition 
    # When arc length is less than R_perturbation it is phi_0 + dphi, 
    # Otherwise, it is just phi_0
    # Here we also ensure we are only looking at positive latitudes (>pi/4 = 45 Deg N)
    w[kIGP] = phi
    w[kIGP][torch.logical_and(gc_dist < radius, lat > np.pi / 4.0)] += dphi

    # We also set the horizontal/angular velocities to be 0 
    w[kIV2] = 0.0
    w[kIV3] = 0.0

    # Return the initial condition as a dictionary
    return {"hydro_w": w}

<div class="alert alert-info">

## Setting Condition and Running Simulation 

This function (the main function) combined many steps we split in previous notebooks.

It loads in the yaml file and creates an output directory. 

It loads in the device (cpu or gpu).

It creates the MeshBlock using the function above and initializes it. 

It starts integrating the model. 

We will comment the below, but many steps can be taken as-is. 

`Set the initial condition into block`

```python
# Create an empty block variables dictionary
block_vars = {}

# Populate the primitive variables with the initial condition we created above
block_vars["hydro_w"] = w

# Initialize the MeshBlock (block) object with the block variables 
block_vars, current_time = block.initialize(block_vars)

<div class="alert alert-info">

## Starting the Simulation

Below, we actually start integrating the model.

The code below will be more-or-less the same for all $\texttt{PADDLE}$ python scripts, and can be taken as-is for now. 

<div>

```python 

def main() -> None:

    # Either read in the input yaml and output directory from the terminal command
    # Or use the defaults 
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", default="shallow_splash_made_in_notebook.yaml")
    parser.add_argument("--output-dir", default="./splash_results")
    args = parser.parse_args()

    # Generate the output directory, if it doesn't already exist 
    os.makedirs(args.output_dir, exist_ok=True)

    # Read in the yaml file as a dictionary 
    with open(args.input, "r", encoding="utf-8") as stream:
        config = yaml.safe_load(stream)

    # Set hydrodynamic options from our yaml file 
    # Equivalent to op from previous notebooks
    options = MeshOptions.from_yaml(args.input, verbose=False)
    device = torch.device(options.device_str())

    # Set the output directory 
    options.block().output_dir(args.output_dir)

    # Here we create the global Mesh 
    # Think of the Mesh like the entire global domain, the the blocks as individual pieces we run in parallel
    mesh = Mesh(options)

    # Move the entire simulation onto the cpu or gpu device 
    mesh.to(device)

    # Set up the starting condition for each individual sub-grid (block)
    # At this point, it is only setting it up for each block assigned to the processor working on this part of the code 
    block_vars = [initialize_block(block, config, device) for block in mesh.blocks]

    # Initialize the Mesh object with the block variables 
    block_vars, current_time = mesh.initialize(block_vars)

    # Initialize the output before the simulation starts, compute until the desired time 
    mesh.make_outputs(block_vars, current_time)

    # Fetch the simulation time integrator module belonging to the first mesh sub-block
    intg = mesh.module("block0.intg")
    # Initialize the frame/step cycle tracking counter to zero
    cycle = 0

    # Run the simulation in a loop
    while not intg.stop(cycle, current_time):

        # Step up the cycle count tracking int, and sync to mesh 
        cycle += 1
        mesh.set_cycle(cycle)

        # Each time step of the simulation is determined by the cfl number and the sound speed 
        dt = mesh.max_time_step(block_vars)

        # Output to let us know the code is running
        mesh.print_cycle_info(block_vars, current_time, dt)

        # For each cycle, a multi-stage method (here rk3) is used to advance block_vars
        for stage in range(len(intg.stages)):
            mesh.forward(block_vars, dt, stage)

        # Check for any errors
        err = mesh.check_redo(block_vars)
        if err > 0:
            continue # redo current step
        if err < 0:
            break   # terminatate
        
        # Progress the time and make outputs
        current_time += dt
        mesh.make_outputs(block_vars, current_time)

    # Make the final outputs and clean up the internal states in mesh
    mesh.finalize(block_vars, current_time)

# Safeguard to ensure this code executes only when launched directly (and not when imported elsewhere)
if __name__ == "__main__":
    main()

<div class="alert alert-info">

## Step-by-step running Splash in terminal


1. Open a terminal and activate your $\texttt{PADDLE}$ conda environment
2. `cd` into the directory storing the `shallow_splash.yaml` and `shallow_splash.py` (the directory with this notebook in it)
3. Run the following code in terminal (for cubed-sphere geometries, 6 = number of cores = 6 * nb2^2)
    ```
    BACKEND=gloo DEVICE=cpu torchrun --nproc-per-node=6 shallow_splash.py
    ```
4. Note, you can also over-ride the default .yaml and output directory defined in the .py script in the terminal command (which is why we import argparse) via 
    ```
    BACKEND=gloo DEVICE=cpu torchrun --nproc-per-node=6 shallow_splash.py --input x.yaml --output-dir ./y_outputs
    ```

`Why is it out0 instead of 1?` 
</div>

<div class="alert alert-block alert-success">

On we go to **Part 4**, where we view and analyze the results in a notebook. 